
# Train tiếp model ngữ cảnh tiếng Việt (2CongLC-IME)

Notebook này **train tiếp** (warm-start) model ONNX ngữ cảnh dùng cho tính năng
"gợi ý theo ngữ cảnh" của bộ gõ 2CongLC-IME — model nhỏ (embed 2 âm tiết trước
→ dự đoán âm tiết tiếp theo), dùng để phát hiện lỗi nhầm âm/dấu kiểu "chia sẻ"
↔ "chia sẽ" mà từ điển đơn thuần không bắt được.

**Trước khi chạy:**
1. Vào `Runtime > Change runtime type`, chọn GPU (không bắt buộc — model rất
   nhỏ nên CPU cũng chạy được, nhưng GPU giúp train nhiều epoch nhanh hơn).
2. Chuẩn bị sẵn 2 file từ project: `vocab.json` và `nplm.pt` (checkpoint hiện
   tại) — sẽ upload ở bước dưới.

**Sau khi chạy xong:** tải file `vn_context_lm.onnx` mới ở cell cuối, gửi lại
để cập nhật vào project (thay file cũ, giữ nguyên `vn_vocab.txt` vì vocab
không đổi).


## 1. Cài thư viện

In [ ]:

!pip install -q onnx onnxscript
import torch
print("torch:", torch.__version__)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 2. Upload checkpoint hiện tại (`vocab.json` và `nplm.pt`)
Lấy 2 file này từ project (đã export ở phiên trước).

In [ ]:

from google.colab import files
print("Chọn vocab.json và nplm.pt:")
uploaded = files.upload()
assert "vocab.json" in uploaded, "Thiếu vocab.json"
assert "nplm.pt" in uploaded, "Thiếu nplm.pt"


## 3. Tải corpus Wikipedia tiếng Việt (nguồn dữ liệu train)
Cùng nguồn đã dùng ở phiên train trước, để dữ liệu nhất quán. Repo ~150MB, có thể mất vài phút.

In [ ]:

!git clone --depth 1 https://github.com/undertheseanlp/corpus.viwiki.git


## 4. Ghép + tokenize corpus (giống hệt cách đã làm ở phiên trước)
**Quan trọng**: dùng lại `vocab.json` đã upload (không xây vocab mới) để checkpoint cũ tương thích với dữ liệu mới.

In [ ]:

import glob, re, json, numpy as np

files_list = glob.glob("corpus.viwiki/viwiki/*.txt")
print("số file:", len(files_list))

texts = []
for fp in files_list:
    try:
        with open(fp, encoding="utf-8", errors="ignore") as f:
            texts.append(f.read())
    except Exception:
        pass
raw = "\n".join(texts)
print("số ký tự:", len(raw))

cleaned = re.sub(r"[^\w\s]|_", " ", raw, flags=re.UNICODE)
cleaned = re.sub(r"\d+", " ", cleaned)
tokens = cleaned.lower().split()
print("số token:", len(tokens))

with open("vocab.json", encoding="utf-8") as f:
    word2id = json.load(f)
vocab_size = len(word2id)
UNK = word2id["<unk>"]
print("vocab_size:", vocab_size)

ids = np.array([word2id.get(t, UNK) for t in tokens], dtype=np.int32)
print("đã mã hoá:", len(ids), "token")


## 5. Định nghĩa model (kiến trúc phải khớp checkpoint cũ) + load warm-start

In [ ]:

import torch, torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

emb_dim, hidden = 48, 128

class NPLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.fc1 = nn.Linear(emb_dim*2, hidden)
        self.fc2 = nn.Linear(hidden, vocab_size)
    def forward(self, c):
        e = self.embed(c).view(c.size(0), -1)
        h = torch.relu(self.fc1(e))
        return self.fc2(h)

model = NPLM().to(device)
model.load_state_dict(torch.load("nplm.pt", map_location=device))
print("Đã load checkpoint cũ — train tiếp (warm-start), không train lại từ đầu.")


## 6. Train tiếp
Dùng **toàn bộ** dữ liệu (không subsample thưa như phiên trước) và nhiều epoch hơn — vì Colab không bị giới hạn thời gian mỗi lệnh như trước. Có thể chỉnh `EPOCHS` tuỳ thời gian bạn muốn chờ (mỗi epoch full data trên GPU T4 thường vài phút; trên CPU lâu hơn nhiều, có thể giảm bớt bằng cách tăng `STRIDE`).

In [ ]:

import time

STRIDE = 1      # 1 = dùng hết dữ liệu; tăng lên (2, 4...) nếu muốn train nhanh hơn
EPOCHS = 15
BATCH = 8192 if device.type == "cuda" else 4096

N = len(ids)
idx = np.arange(2, N, STRIDE)
ctx = np.stack([ids[idx-2], ids[idx-1]], axis=1).astype(np.int64)
tgt = ids[idx].astype(np.int64)
print("số mẫu train:", len(tgt))

n_val = 200_000
ctx_train, tgt_train = ctx[:-n_val], tgt[:-n_val]
ctx_val, tgt_val = ctx[-n_val:], tgt[-n_val:]

ctx_train_t = torch.from_numpy(ctx_train).to(device)
tgt_train_t = torch.from_numpy(tgt_train).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

n_train = len(tgt_train)
steps_per_epoch = n_train // BATCH
print("steps/epoch:", steps_per_epoch)

t0 = time.time()
for epoch in range(EPOCHS):
    perm = torch.randperm(n_train, device=device)
    total_loss = 0.0
    for i in range(steps_per_epoch):
        b = perm[i*BATCH:(i+1)*BATCH]
        c, t = ctx_train_t[b], tgt_train_t[b]
        opt.zero_grad()
        out = model(c)
        loss = lossfn(out, t)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        if i % 500 == 0:
            print(f"epoch {epoch} step {i}/{steps_per_epoch} loss {loss.item():.3f} elapsed {time.time()-t0:.0f}s")
    print(f"== epoch {epoch} avg loss {total_loss/steps_per_epoch:.3f} ==")

torch.save(model.state_dict(), "nplm.pt")
print("Tổng thời gian train:", time.time()-t0, "s")


## 7. Kiểm tra nhanh độ chính xác + vài cặp dễ nhầm

In [ ]:

model.eval()
with torch.no_grad():
    correct = 0
    vb = 8192
    for i in range(0, len(tgt_val), vb):
        c = torch.from_numpy(ctx_val[i:i+vb]).to(device)
        t = torch.from_numpy(tgt_val[i:i+vb]).to(device)
        pred = model(c).argmax(dim=1)
        correct += (pred == t).sum().item()
    print("val top-1 acc:", correct/len(tgt_val))

id2word = {i: w for w, i in word2id.items()}

def prob(prev2, prev1, word):
    c1 = word2id.get(prev2, UNK)
    c2 = word2id.get(prev1, UNK)
    ctx_t = torch.tensor([[c1, c2]], dtype=torch.long, device=device)
    with torch.no_grad():
        p = torch.softmax(model(ctx_t), dim=1)[0]
    wid = word2id.get(word, UNK)
    return p[wid].item()

tests = [
    ("con", "dao", "sắc"), ("con", "dao", "xắc"),
    ("tôi", "sẽ", "sửa"), ("tôi", "sẽ", "sữa"),
    ("chúng", "ta", "chia"),
]
for a, b, w in tests:
    print(f"{a} {b} __{w}__ -> p={prob(a,b,w):.6f}")


## 8. Export ONNX + tải về

In [ ]:

class NPLMSoftmax(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
    def forward(self, c):
        return torch.softmax(self.base(c), dim=1)

export_model = NPLMSoftmax(model).cpu().eval()
dummy = torch.zeros((1, 2), dtype=torch.long)

torch.onnx.export(
    export_model, dummy, "vn_context_lm.onnx",
    input_names=["context"], output_names=["probs"],
    dynamic_axes={"context": {0: "batch"}, "probs": {0: "batch"}},
    opset_version=13,
    dynamo=False,
)
print("Đã export vn_context_lm.onnx")

from google.colab import files
files.download("vn_context_lm.onnx")
files.download("nplm.pt")  # để lần sau train tiếp nữa nếu muốn


## Xong
Gửi file **`vn_context_lm.onnx`** vừa tải về lại cho Claude để cập nhật vào project (thay file cũ trong thư mục project, giữ nguyên `vn_vocab.txt` vì vocab không đổi). Giữ lại `nplm.pt` nếu muốn train tiếp thêm ở phiên sau.